In [1]:
import sys
sys.path.append("../")

import pandas as pd
import numpy as np

from src.data_loader import load_csv
from src.preprocessing import remove_duplicates, convert_datetime_columns, convert_column_to_int
from src.geolocation import merge_ip_country
from src.feature_engineering import add_time_features, add_transaction_features
from src.transformation import prepare_training_data
from src.modeling import (
    train_logistic_regression,
    train_random_forest,
    evaluate_model,
    run_cross_validation,
    save_model
)

In [2]:
fraud = load_csv("../data/raw/Fraud_Data.csv")
ip_country = load_csv("../data/raw/IpAddress_to_Country.csv")

fraud = remove_duplicates(fraud)
fraud = convert_datetime_columns(fraud, ["signup_time", "purchase_time"])
fraud = convert_column_to_int(fraud, "ip_address")

fraud_geo = merge_ip_country(fraud, ip_country)
fraud_geo = add_time_features(fraud_geo)
fraud_geo = add_transaction_features(fraud_geo)

X_train_fraud, X_test_fraud, y_train_fraud, y_test_fraud = prepare_training_data(
    fraud_geo,
    target_column="class",
    drop_columns=["signup_time", "purchase_time", "device_id", "user_id"]
)

Before SMOTE:
class
0    109568
1     11321
Name: count, dtype: int64
After SMOTE:
class
0    109568
1    109568
Name: count, dtype: int64


In [3]:
fraud_lr = train_logistic_regression(X_train_fraud, y_train_fraud)

fraud_rf = train_random_forest(
    X_train_fraud,
    y_train_fraud,
    n_estimators=100,
    max_depth=10
)

In [4]:
fraud_lr_results = evaluate_model(
    fraud_lr,
    X_test_fraud,
    y_test_fraud,
    "Fraud Data - Logistic Regression"
)

fraud_rf_results = evaluate_model(
    fraud_rf,
    X_test_fraud,
    y_test_fraud,
    "Fraud Data - Random Forest"
)


Fraud Data - Logistic Regression Results
AUC-PR: 0.6664975361241878
F1 Score: 0.6042095268238645
Confusion Matrix:
[[25813  1580]
 [  921  1909]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.94      0.95     27393
           1       0.55      0.67      0.60      2830

    accuracy                           0.92     30223
   macro avg       0.76      0.81      0.78     30223
weighted avg       0.93      0.92      0.92     30223


Fraud Data - Random Forest Results
AUC-PR: 0.7041256752408466
F1 Score: 0.6043613707165109
Confusion Matrix:
[[25743  1650]
 [  890  1940]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.94      0.95     27393
           1       0.54      0.69      0.60      2830

    accuracy                           0.92     30223
   macro avg       0.75      0.81      0.78     30223
weighted avg       0.93      0.92      0.92     30223



In [5]:
credit = load_csv("../data/raw/creditcard.csv")
credit = remove_duplicates(credit)
credit = convert_column_to_int(credit, "Class")

X_train_credit, X_test_credit, y_train_credit, y_test_credit = prepare_training_data(
    credit,
    target_column="Class"
)

Before SMOTE:
Class
0    226602
1       378
Name: count, dtype: int64
After SMOTE:
Class
0    226602
1    226602
Name: count, dtype: int64


In [6]:
credit_lr = train_logistic_regression(X_train_credit, y_train_credit)

credit_rf = train_random_forest(
    X_train_credit,
    y_train_credit,
    n_estimators=100,
    max_depth=10
)

In [7]:
credit_lr_results = evaluate_model(
    credit_lr,
    X_test_credit,
    y_test_credit,
    "Credit Card - Logistic Regression"
)

credit_rf_results = evaluate_model(
    credit_rf,
    X_test_credit,
    y_test_credit,
    "Credit Card - Random Forest"
)


Credit Card - Logistic Regression Results
AUC-PR: 0.6750409387333339
F1 Score: 0.1
Confusion Matrix:
[[55169  1482]
 [   12    83]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.99     56651
           1       0.05      0.87      0.10        95

    accuracy                           0.97     56746
   macro avg       0.53      0.92      0.54     56746
weighted avg       1.00      0.97      0.99     56746


Credit Card - Random Forest Results
AUC-PR: 0.7762801845570437
F1 Score: 0.6610169491525424
Confusion Matrix:
[[56588    63]
 [   17    78]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.55      0.82      0.66        95

    accuracy                           1.00     56746
   macro avg       0.78      0.91      0.83     56746
weighted avg       1.00      1.00      1.00     56746



In [8]:
results_df = pd.DataFrame([
    {
        "Dataset": "Fraud_Data",
        "Model": "Logistic Regression",
        "AUC_PR": fraud_lr_results["auc_pr"],
        "F1_Score": fraud_lr_results["f1_score"]
    },
    {
        "Dataset": "Fraud_Data",
        "Model": "Random Forest",
        "AUC_PR": fraud_rf_results["auc_pr"],
        "F1_Score": fraud_rf_results["f1_score"]
    },
    {
        "Dataset": "Credit Card",
        "Model": "Logistic Regression",
        "AUC_PR": credit_lr_results["auc_pr"],
        "F1_Score": credit_lr_results["f1_score"]
    },
    {
        "Dataset": "Credit Card",
        "Model": "Random Forest",
        "AUC_PR": credit_rf_results["auc_pr"],
        "F1_Score": credit_rf_results["f1_score"]
    }
])

results_df


,Dataset,Model,AUC_PR,F1_Score
0,Fraud_Data,Logistic Regression,0.666498,0.604210
1,Fraud_Data,Random Forest,0.704126,0.604361
2,Credit Card,Logistic Regression,0.675041,0.100000
3,Credit Card,Random Forest,0.776280,0.661017


In [9]:
results_df.to_csv("../reports/task2_model_comparison.csv", index=False)

In [10]:
save_model(fraud_lr, "../models/fraud_logistic_regression.pkl")
save_model(fraud_rf, "../models/fraud_random_forest.pkl")
save_model(credit_lr, "../models/credit_logistic_regression.pkl")
save_model(credit_rf, "../models/credit_random_forest.pkl")